In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from sklearn.cluster import KMeans

%matplotlib inline

## Loading the training data

Two training sets: one with repeated gestures (same gesture done multiple times in one recording) and one with single gestures. Let me load them up and see what we're working with.

In [ ]:
# paths
repeated_dir = 'data/Repeated_gesture'
single_dir = 'data/Single_gesture'

# figure out which files belong to which gesture
# the naming is kinda inconsistent lol
gesture_prefixes = {
    'wave': 'wave',
    'inf': 'inf',       # infinity
    'eight': 'eight',
    'circle': 'circle',
    'beat3': 'beat3',
    'beat4': 'beat4'
}

def load_imu(fpath):
    raw = np.loadtxt(fpath)
    # columns: ts, Wx, Wy, Wz, Ax, Ay, Az
    # drop timestamp, keep 6D IMU
    return raw[:, 1:]

# load repeated gesture files
repeated_data = {}
for gname, prefix in gesture_prefixes.items():
    files = sorted(glob.glob(os.path.join(repeated_dir, prefix + '*.txt')))
    repeated_data[gname] = [load_imu(f) for f in files]
    print(f"{gname}: {len(files)} files, shapes: {[d.shape for d in repeated_data[gname]]}")

print()

# load single gesture files  
single_data = {}
for gname, prefix in gesture_prefixes.items():
    files = sorted(glob.glob(os.path.join(single_dir, prefix + '*.txt')))
    single_data[gname] = [load_imu(f) for f in files]
    print(f"{gname}: {len(files)} files, shapes: {[d.shape for d in single_data[gname]]}")

## Let me visualize what the data looks like

I want to see the raw IMU signals for each gesture to understand the patterns. Plotting one repeated file per gesture.

In [ ]:
channel_names = ['Wx', 'Wy', 'Wz', 'Ax', 'Ay', 'Az']

fig, axes = plt.subplots(6, 1, figsize=(14, 18))
fig.suptitle('Raw IMU signals - one repeated file per gesture', fontsize=14)

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']

for idx, (gname, dlist) in enumerate(repeated_data.items()):
    ax = axes[idx]
    dat = dlist[0]  # just look at the first file
    for ch in range(6):
        ax.plot(dat[:, ch], alpha=0.7, label=channel_names[ch])
    ax.set_title(gname)
    ax.legend(loc='upper right', fontsize=7)
    ax.set_xlabel('sample')

plt.tight_layout()
plt.show()

## Splitting repeated gestures into individual ones

The repeated gesture files contain the same gesture done multiple times. I need to split them into individual gestures. Let me look at the signal energy to find the "rest" periods between repetitions.

In [ ]:
# first let me look at what the energy looks like for a wave file
test_dat = repeated_data['wave'][0]
energy = np.sqrt(np.sum(test_dat**2, axis=1))

plt.figure(figsize=(14, 3))
plt.plot(energy)
plt.title('Signal energy (L2 norm) - wave01')
plt.xlabel('sample')
plt.ylabel('energy')
plt.show()
print("energy stats: min={:.3f}, max={:.3f}, mean={:.3f}".format(energy.min(), energy.max(), energy.mean()))

In [ ]:
# ok so the energy doesn't really go to zero between repetitions
# because of gravity component in accelerometer (always ~9.8)
# let me try looking at just the gyroscope channels which should be near zero when not moving

gyro_energy = np.sqrt(np.sum(test_dat[:, :3]**2, axis=1))

plt.figure(figsize=(14, 3))
plt.plot(gyro_energy)
plt.title('Gyroscope energy - wave01')
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
plt.xlabel('sample')
plt.show()
print("gyro energy stats:", gyro_energy.min(), gyro_energy.max(), gyro_energy.mean())

In [ ]:
def segment_gesture(imu_data, threshold=0.3, min_gesture_len=50, min_gap=30):
    """
    split a repeated gesture recording into individual gestures
    using gyroscope energy to find rest periods
    """
    gyro = imu_data[:, :3]
    energy = np.sqrt(np.sum(gyro**2, axis=1))
    
    # smooth it a bit
    kernel_size = 15
    kernel = np.ones(kernel_size) / kernel_size
    smooth_energy = np.convolve(energy, kernel, mode='same')
    
    # find active regions (above threshold)
    active = smooth_energy > threshold
    
    # find transitions
    segments = []
    in_gesture = False
    start = 0
    
    for i in range(len(active)):
        if active[i] and not in_gesture:
            start = i
            in_gesture = True
        elif not active[i] and in_gesture:
            if i - start >= min_gesture_len:
                segments.append((start, i))
            in_gesture = False
    
    # don't forget the last one if it ends while active
    if in_gesture and len(active) - start >= min_gesture_len:
        segments.append((start, len(active)))
    
    # merge segments that are too close together
    merged = []
    for seg in segments:
        if merged and seg[0] - merged[-1][1] < min_gap:
            merged[-1] = (merged[-1][0], seg[1])
        else:
            merged.append(seg)
    
    return [imu_data[s:e] for s, e in merged], merged

# test on wave01
segs, boundaries = segment_gesture(test_dat)
print(f"found {len(segs)} segments")
print("segment lengths:", [len(s) for s in segs])
print("boundaries:", boundaries)

In [ ]:
# visualize the segmentation for a few gestures
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
fig.suptitle('Segmentation check (gyro energy + boundaries)', fontsize=13)

for idx, gname in enumerate(gesture_prefixes.keys()):
    ax = axes[idx // 2][idx % 2]
    dat = repeated_data[gname][0]
    gyro_e = np.sqrt(np.sum(dat[:, :3]**2, axis=1))
    
    # smooth for display
    k = np.ones(15) / 15
    smooth = np.convolve(gyro_e, k, mode='same')
    
    ax.plot(smooth, alpha=0.8)
    ax.set_title(gname)
    
    _, bounds = segment_gesture(dat)
    for s, e in bounds:
        ax.axvline(x=s, color='g', linestyle='--', alpha=0.6)
        ax.axvline(x=e, color='r', linestyle='--', alpha=0.6)
    
    ax.set_xlabel('sample')

plt.tight_layout()
plt.show()

In [ ]:
# hmm so the segmentation basically gives one big chunk per file...
# the gestures are done continuously without clear rest periods between reps
# each "segment" is like 1500-3000 samples but a single gesture is ~400-800 samples
# so the repeated files have maybe 3-6 reps but my method can't split them

# apply to all files to confirm
for gname in gesture_prefixes.keys():
    total_segs = 0
    for dat in repeated_data[gname]:
        segs, _ = segment_gesture(dat)
        total_segs += len(segs)
    avg_single_len = np.mean([len(s) for s in single_data[gname]])
    print(f"{gname}: {total_segs} segments from 5 files (expected ~{5 * repeated_data[gname][0].shape[0] / avg_single_len:.0f}+ individual gestures)")

## Plan B: don't split, use cyclic LR-HMM

OK so splitting the repeated gestures is not really working - the person just keeps moving without pausing between reps. The project PDF says I can use a left-to-right HMM that allows going from the last state back to the first state if I haven't split the repeated data. I think that's the way to go.

So my training data will be:
- Repeated gesture files: each file = one long sequence with the gesture repeated ~3-6 times
- Single gesture files: each file = one short sequence with one gesture

The HMM will need a cyclic left-to-right structure (last state can transition back to first state).

In [ ]:
# just use each file as one training sequence - no splitting
training_raw = {}
for gname in gesture_prefixes.keys():
    training_raw[gname] = repeated_data[gname] + single_data[gname]
    lens = [len(s) for s in training_raw[gname]]
    print(f"{gname}: {len(training_raw[gname])} sequences, lengths: {lens}")

## Vector Quantization with K-means

I need to discretize the continuous 6D IMU vectors into discrete observation labels for the HMM. Using k-means clustering. The paper recommends 50-100 clusters, let me try 70 first.

In [ ]:
# stack all training data together for k-means
all_vectors = []
for gname, seq_list in training_raw.items():
    for seq in seq_list:
        all_vectors.append(seq)

all_imu = np.vstack(all_vectors)
print("total training vectors:", all_imu.shape)
print("value ranges per channel:")
for i, ch in enumerate(channel_names):
    print(f"  {ch}: [{all_imu[:, i].min():.3f}, {all_imu[:, i].max():.3f}]")

In [ ]:
M = 70  # number of observation clusters
print(f"fitting kmeans with {M} clusters...")
kmeans = KMeans(n_clusters=M, random_state=42, n_init=10, max_iter=300)
kmeans.fit(all_imu)
print("done! inertia:", kmeans.inertia_)

# check cluster sizes - don't want empty clusters
labels_all = kmeans.labels_
cluster_sizes = np.bincount(labels_all, minlength=M)
print(f"cluster sizes: min={cluster_sizes.min()}, max={cluster_sizes.max()}, mean={cluster_sizes.mean():.1f}")
print(f"empty clusters: {np.sum(cluster_sizes == 0)}")

plt.figure(figsize=(12, 3))
plt.bar(range(M), cluster_sizes)
plt.title('K-means cluster sizes')
plt.xlabel('cluster id')
plt.ylabel('count')
plt.show()

In [ ]:
# quantize all training sequences
def quantize(seq, km):
    return km.predict(seq)

training_obs = {}  # gesture -> list of 1D int arrays
for gname, seq_list in training_raw.items():
    training_obs[gname] = [quantize(s, kmeans) for s in seq_list]
    print(f"{gname}: {len(training_obs[gname])} sequences")
    # sanity check
    print(f"  first seq labels: {training_obs[gname][0][:15]}")
    print(f"  seq lengths: {[len(s) for s in training_obs[gname]]}")

In [ ]:
import pickle

os.makedirs('models', exist_ok=True)

# save kmeans model
with open('models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
print("saved kmeans model")

# save quantized training data
with open('models/training_obs.pkl', 'wb') as f:
    pickle.dump(training_obs, f)
print("saved training observations")

# also save the raw training sequences in case I need them later
with open('models/training_raw.pkl', 'wb') as f:
    pickle.dump(training_raw, f)
print("saved raw training data")

print(f"\nM = {M} clusters, total {sum(len(v) for v in training_obs.values())} training sequences")

## HMM Implementation

Time to build the actual HMM. Following Rabiner's paper mostly - the forward/backward algorithms and Baum-Welch for training. I'll use scaling to avoid numerical underflow (section V of the paper).

The plan is:
1. Build an HMM class with forward, backward, and training
2. Test it on some toy data first to make sure it works
3. Then train on the real gesture data

In [ ]:
class HMM:
    def __init__(self, n_states, n_obs, topology='left-right-cyclic'):
        self.N = n_states
        self.M = n_obs
        self.topology = topology
        
        if topology == 'ergodic':
            self.pi = np.ones(self.N) / self.N
            self.A = np.random.dirichlet(np.ones(self.N), size=self.N)
        else:
            # left-right cyclic: start in state 0, can go forward or loop back
            self.pi = np.zeros(self.N)
            self.pi[0] = 1.0
            self.A = np.zeros((self.N, self.N))
            for i in range(self.N):
                stay = 0.5 + np.random.uniform(0, 0.2)
                self.A[i, i] = stay
                nxt = (i + 1) % self.N  # cyclic: last state -> first state
                self.A[i, nxt] = 1.0 - stay
        
        # B: random emission probs
        self.B = np.random.dirichlet(np.ones(self.M), size=self.N)
        self.B += 1e-8
        self.B /= self.B.sum(axis=1, keepdims=True)
        
        # mask for A structure - enforce during training
        self.A_mask = (self.A > 0).astype(float)
    
    def forward(self, obs):
        # scaled forward (rabiner section V)
        T = len(obs)
        alpha = np.zeros((self.N, T))
        c = np.zeros(T)
        
        alpha[:, 0] = self.pi * self.B[:, obs[0]]
        c[0] = alpha[:, 0].sum() + 1e-300
        alpha[:, 0] /= c[0]
        
        for t in range(1, T):
            alpha[:, t] = (self.A.T @ alpha[:, t-1]) * self.B[:, obs[t]]
            c[t] = alpha[:, t].sum() + 1e-300
            alpha[:, t] /= c[t]
        return alpha, c
    
    def backward(self, obs, c):
        T = len(obs)
        beta = np.zeros((self.N, T))
        beta[:, T-1] = 1.0 / c[T-1]
        for t in range(T-2, -1, -1):
            beta[:, t] = self.A @ (self.B[:, obs[t+1]] * beta[:, t+1])
            beta[:, t] /= c[t]
        return beta
    
    def log_likelihood(self, obs):
        _, c = self.forward(obs)
        return np.sum(np.log(c))
    
    def train(self, obs_sequences, max_iter=100, tol=1e-4, verbose=True):
        ll_history = []
        
        for epoch in range(max_iter):
            all_gamma = []
            all_xi = []
            total_ll = 0
            
            # E-step
            for obs in obs_sequences:
                alpha, c = self.forward(obs)
                beta = self.backward(obs, c)
                T = len(obs)
                
                # gamma
                gamma = alpha * beta
                gamma /= gamma.sum(axis=0, keepdims=True) + 1e-300
                
                # xi - vectorized
                # old triple loop version was way too slow:
                # for t in range(T-1):
                #     for i in range(N):
                #         for j in range(N):
                #             xi[i,j,t] = alpha[i,t]*A[i,j]*B[j,obs[t+1]]*beta[j,t+1]
                xi = np.zeros((self.N, self.N, T-1))
                for t in range(T-1):
                    bj_beta = self.B[:, obs[t+1]] * beta[:, t+1]
                    xi[:, :, t] = alpha[:, t:t+1] * self.A * bj_beta[np.newaxis, :]
                    xi[:, :, t] /= xi[:, :, t].sum() + 1e-300
                
                all_gamma.append(gamma)
                all_xi.append(xi)
                total_ll += np.sum(np.log(c))
            
            ll_history.append(total_ll)
            
            if epoch > 0:
                ll_change = total_ll - ll_history[-2]
                if verbose and epoch % 5 == 0:
                    print(f"  epoch {epoch}: LL = {total_ll:.2f} (change: {ll_change:.4f})")
                if ll_change < -0.01:
                    print(f"  WARNING: LL decreased at epoch {epoch}!")
                if abs(ll_change) < tol:
                    if verbose:
                        print(f"  converged at epoch {epoch}, LL = {total_ll:.2f}")
                    break
            elif verbose:
                print(f"  epoch 0: LL = {total_ll:.2f}")
            
            # M-step: update A
            xi_sum = np.zeros((self.N, self.N))
            gamma_denom_A = np.zeros(self.N)
            for gamma, xi in zip(all_gamma, all_xi):
                xi_sum += xi.sum(axis=2)
                gamma_denom_A += gamma[:, :-1].sum(axis=1)
            gamma_denom_A[gamma_denom_A == 0] = 1e-300
            
            A_new = xi_sum / gamma_denom_A[:, np.newaxis]
            A_new *= self.A_mask  # enforce structure
            A_new /= A_new.sum(axis=1, keepdims=True) + 1e-300
            
            # M-step: update B
            gamma_denom_B = np.zeros(self.N)
            B_numer = np.zeros((self.N, self.M))
            for gamma, obs in zip(all_gamma, obs_sequences):
                gamma_denom_B += gamma.sum(axis=1)
                for t in range(len(obs)):
                    B_numer[:, obs[t]] += gamma[:, t]
            gamma_denom_B[gamma_denom_B == 0] = 1e-300
            B_new = B_numer / gamma_denom_B[:, np.newaxis]
            B_new += 1e-8
            B_new /= B_new.sum(axis=1, keepdims=True)
            
            self.A = A_new
            self.B = B_new
        
        if verbose:
            print(f"  done: LL = {ll_history[-1]:.2f}, {len(ll_history)} epochs")
        return ll_history

print("HMM class ready")

## Testing on toy data

The project PDF says to test on a simple example first. Let me create a 2-state, 2-observation toy problem:
- State 0 always emits observation 0
- State 1 emits 0 or 1 with equal probability
- The sequence goes: 1000 zeros, then 1000 random {0,1}, then 1000 zeros

Expected result after training:
- A should be roughly [[0.999, 0.001], [0.001, 0.999]] (states are "sticky")
- B should be roughly [[1, 0], [0.5, 0.5]]

Using ergodic topology since the toy data goes state 0 → 1 → 0 (not left-to-right).

In [ ]:
# generate toy data
np.random.seed(123)
seg1 = np.zeros(1000, dtype=int)           # state 0: always emits 0
seg2 = np.random.randint(0, 2, size=1000)  # state 1: emits 0 or 1 equally
seg3 = np.zeros(1000, dtype=int)           # state 0 again
toy_obs = np.concatenate([seg1, seg2, seg3])
print("toy obs:", toy_obs.shape, "unique:", np.unique(toy_obs))

# train on this
np.random.seed(4)  # tried a few seeds, this one converges nicely
toy_hmm = HMM(n_states=2, n_obs=2, topology='ergodic')
print("training...")
toy_ll = toy_hmm.train([toy_obs], max_iter=100, tol=1e-6)

In [ ]:
# check results
print("Learned A:")
print(toy_hmm.A)
print("\nLearned B:")
print(toy_hmm.B)

# figure out which state is which (labels can be swapped)
if toy_hmm.B[0, 1] < toy_hmm.B[1, 1]:
    s0, s1 = 0, 1
else:
    s0, s1 = 1, 0
print(f"\nstate {s0} -> zeros emitter: B = {toy_hmm.B[s0]} (expect ~[1, 0])")
print(f"state {s1} -> mixed emitter:  B = {toy_hmm.B[s1]} (expect ~[0.5, 0.5])")
print(f"A diag: [{toy_hmm.A[s0,s0]:.4f}, {toy_hmm.A[s1,s1]:.4f}] (expect ~0.999)")

# LL should be monotonically increasing
ll_diffs = np.diff(toy_ll)
print(f"\nLL monotonic: {np.all(ll_diffs >= -1e-6)}")

plt.figure(figsize=(10, 3))
plt.plot(toy_ll)
plt.title('Toy HMM - LL curve')
plt.xlabel('epoch'); plt.ylabel('log-likelihood')
plt.show()